# Fine-tuning & Prompt Optimization

#### Fine-tuning vs Prompting — The Decision

PROMPTING (what you've done so far):
  ✅ No training needed
  ✅ Change behavior instantly
  ✅ Works with any model
  ❌ Limited by base model knowledge
  ❌ Long prompts = expensive/slow
  ❌ Inconsistent on edge cases

FINE-TUNING:
  ✅ Model learns YOUR style/format
  ✅ Shorter prompts needed
  ✅ Better on domain-specific tasks
  ✅ Faster inference (smaller prompts)
  ❌ Takes time and data to train
  ❌ Need 100-1000+ examples
  ❌ Expensive to retrain often

WHEN TO FINE-TUNE:
  ✅ Consistent output format always needed
  ✅ Domain-specific language (medical, legal)
  ✅ Style matching (your company's tone)
  ✅ Task prompting costs > fine-tuning cost
  ❌ You have < 100 examples
  ❌ Task changes frequently
  ❌ General Q&A (prompting works fine)

In [2]:
# Cell 1: Imports
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json, os, time
from typing import List, Dict, Optional
from dotenv import load_dotenv

load_dotenv()
llm  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
fast = ChatGroq(model="llama-3.1-8b-instant",    temperature=0)

print("✅ Ready")

d:\AI\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Ready


#### Prepare Fine-tuning Dataset

In [3]:
# Cell 2: Create a training dataset
# Format: instruction → response pairs
# We'll create a dataset for a customer support bot

training_examples = [
    {
        "instruction": "Customer says: My order hasn't arrived yet. Order #12345",
        "response": "I apologize for the delay. Let me check order #12345 for you right away. Could you please confirm your email address so I can pull up the details and provide you with an accurate update on your delivery status?"
    },
    {
        "instruction": "Customer says: I want to return this product",
        "response": "I'd be happy to help you with your return. Our return policy allows returns within 30 days of purchase. Please provide your order number and reason for return, and I'll initiate the process immediately."
    },
    {
        "instruction": "Customer says: The product is broken",
        "response": "I'm sorry to hear the product arrived damaged. This is not the experience we want for our customers. Please share a photo of the damage and your order number, and we'll arrange a replacement or full refund right away."
    },
    {
        "instruction": "Customer says: How do I track my order?",
        "response": "You can track your order in three ways: (1) Check the tracking link in your confirmation email, (2) Log into your account and visit 'My Orders', or (3) Reply here with your order number and I'll get the latest update for you."
    },
    {
        "instruction": "Customer says: I was charged twice",
        "response": "I sincerely apologize for the double charge — that should never happen. I'm escalating this to our billing team immediately. Please share your order number and the last 4 digits of the card charged, and we'll process a refund within 3-5 business days."
    },
    {
        "instruction": "Customer says: Cancel my subscription",
        "response": "I understand you'd like to cancel. Before I process that, may I ask what prompted this decision? If there's something we can improve, I'd love to help. If you'd still like to proceed, I can cancel immediately with no further charges."
    },
    {
        "instruction": "Customer says: The app keeps crashing",
        "response": "I'm sorry the app is giving you trouble. Let's fix this quickly. Could you tell me: (1) Your device model and OS version, (2) When the crashes started, (3) What you were doing when it crashed? This will help me get you the right solution."
    },
    {
        "instruction": "Customer says: I never received my refund",
        "response": "I apologize for this delay. Refunds typically take 5-7 business days to appear. Let me check the status of your refund right now. Could you share your order number and the email associated with your account?"
    },
]

# Convert to Alpaca format (standard fine-tuning format)
def to_alpaca_format(examples: List[Dict]) -> List[Dict]:
    """Convert to standard Alpaca instruction format"""
    formatted = []
    for ex in examples:
        formatted.append({
            "instruction": ex["instruction"],
            "input":       "",   # no additional input
            "output":      ex["response"],
            "text": f"""Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{ex['instruction']}

### Response:
{ex['response']}"""
        })
    return formatted


alpaca_data = to_alpaca_format(training_examples)

# Save dataset
os.makedirs("fine_tune_data", exist_ok=True)
with open("fine_tune_data/customer_support.json", "w") as f:
    json.dump(alpaca_data, f, indent=2)

print(f"✅ Dataset created: {len(alpaca_data)} examples")
print(f"   Format: Alpaca instruction format")
print(f"   Saved:  fine_tune_data/customer_support.json")
print(f"\nSample:")
print(alpaca_data[0]["text"][:300])

✅ Dataset created: 8 examples
   Format: Alpaca instruction format
   Saved:  fine_tune_data/customer_support.json

Sample:
Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
Customer says: My order hasn't arrived yet. Order #12345

### Response:
I apologize for the delay. Let me check order #12345 for you right away. Could you please confirm your e


#### QLoRA Fine-tuning on RTX 5060

In [4]:
# Cell 3: QLoRA setup - quantized LoRA for consumer GPUs

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
from trl import SFTTrainer
from datasets import Dataset

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — will use CPU (slower)")

GPU available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 GB


In [5]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"CUDA capability: sm_{torch.cuda.get_device_capability(0)[0]}{torch.cuda.get_device_capability(0)[1]}")
print(f"GPU:             {torch.cuda.get_device_name(0)}")
print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Quick compute test
x = torch.tensor([1.0, 2.0, 3.0]).cuda()
print(f"\nGPU tensor test: {x * 2}")
print("✅ RTX 5060 working with PyTorch!")

PyTorch version: 2.12.0.dev20260408+cu128
CUDA available:  True
CUDA capability: sm_120
GPU:             NVIDIA GeForce RTX 5060 Laptop GPU
VRAM:            8.5 GB

GPU tensor test: tensor([2., 4., 6.], device='cuda:0')
✅ RTX 5060 working with PyTorch!


In [6]:
# Cell 4: QLoRA configuration
# QLoRA = Quantized LoRA = fine-tune big models on small GPUs

# ── Step 1: Quantization config ───────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # load model in 4-bit
    bnb_4bit_quant_type="nf4",               # NF4 quantization
    bnb_4bit_compute_dtype=torch.float16,    # compute in fp16
    bnb_4bit_use_double_quant=True           # double quantization
)

# ── Step 2: LoRA config ───────────────────────────────────────
lora_config = LoraConfig(
    r=16,                       # rank - higher = more params = better quality
    lora_alpha=32,              # scaling factor (usually 2x rank)
    target_modules=[            # which layers to train
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,          # regularization
    bias="none",                # don't train bias
    task_type=TaskType.CAUSAL_LM
)

print("QLoRA Configuration:")
print(f"""
Quantization:
  bits:          4-bit NF4
  compute dtype: float16
  double quant:  yes

LoRA:
  rank (r):      16
  alpha:         32
  dropout:       0.05
  target:        attention + MLP layers

Memory estimate for Llama-3.2-1B:
  Without QLoRA: ~4GB VRAM
  With QLoRA:    ~1.5GB VRAM  ← your RTX 5060 handles this easily
""")

QLoRA Configuration:

Quantization:
  bits:          4-bit NF4
  compute dtype: float16
  double quant:  yes

LoRA:
  rank (r):      16
  alpha:         32
  dropout:       0.05
  target:        attention + MLP layers

Memory estimate for Llama-3.2-1B:
  Without QLoRA: ~4GB VRAM
  With QLoRA:    ~1.5GB VRAM  ← your RTX 5060 handles this easily



In [7]:
# Cell 5: Load model and apply QLoRA
# Using Llama-3.2-1B - small enough for your 8GB VRAM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
# Alternative if you don't have Llama access:
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # no auth needed

print(f"Loading model: {MODEL_ID}")
print("(First run downloads ~2GB — subsequent runs use cache)\n")

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Load model with 4-bit quantization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",          # auto-detect GPU/CPU
        torch_dtype=torch.float16,
    )

    # Prepare for k-bit training
    model = prepare_model_for_kbit_training(model)

    # Apply LoRA adapters
    model = get_peft_model(model, lora_config)

    # Show trainable parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"✅ Model loaded with QLoRA")
    print(f"   Total params:     {total:,}")
    print(f"   Trainable params: {trainable:,} ({100*trainable/total:.2f}%)")
    print(f"   Frozen params:    {total-trainable:,}")

except Exception as e:
    print(f"⚠️  Model load failed: {e}")
    print("\nFallback: Using TinyLlama (no auth needed)")
    MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    print(f"✅ TinyLlama loaded as fallback")

Loading model: meta-llama/Llama-3.2-1B-Instruct
(First run downloads ~2GB — subsequent runs use cache)

⚠️  Model load failed: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
403 Client Error. (Request ID: Root=1-6a85511c-7774263f173f383f1edcdd9b;6a81681c-03db-405d-af75-3e3711dc6069)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct to ask for access.

Fallback: Using TinyLlama (no auth needed)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:02<00:00, 72.12it/s]
W0819 12:15:57.088000 20432 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


✅ TinyLlama loaded as fallback


In [9]:
# Cell 6: Training
from trl import SFTConfig, SFTTrainer
# Load dataset
with open("fine_tune_data/customer_support.json") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
print(f"Dataset: {len(dataset)} examples")

# Training arguments
training_args = SFTConfig(
    output_dir="./fine_tune_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

# Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,  # replaces tokenizer=
    args=training_args,
    train_dataset=dataset,
)

print("Starting QLoRA fine-tuning...")
print("Expected time: 2-5 minutes on RTX 5060\n")

start = time.time()
trainer.train()
elapsed = time.time() - start

print(f"\n✅ Fine-tuning complete in {elapsed:.0f}s")

# Save the LoRA adapter
model.save_pretrained("./fine_tune_output/lora_adapter")
tokenizer.save_pretrained("./fine_tune_output/lora_adapter")
print("✅ LoRA adapter saved to ./fine_tune_output/lora_adapter")

Dataset: 8 examples


Building labels for train dataset: 100%|██████████| 8/8 [00:00<00:00, 748.77 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting QLoRA fine-tuning...
Expected time: 2-5 minutes on RTX 5060



Step,Training Loss



✅ Fine-tuning complete in 13s
✅ LoRA adapter saved to ./fine_tune_output/lora_adapter


In [10]:
# Cell 7: Test fine-tuned model

def generate_response(prompt: str, max_tokens: int = 200) -> str:
    """Generate response from fine-tuned model"""
    formatted = f"""Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{prompt}

### Response:"""

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Test on new scenarios not in training data
test_prompts = [
    "Customer says: I ordered the wrong size",
    "Customer says: Where is my invoice?",
    "Customer says: The color looks different from the website photo",
]

print("Fine-tuned model responses:\n")
for prompt in test_prompts:
    print(f"Input:    {prompt}")
    response = generate_response(prompt)
    print(f"Response: {response[:200]}")
    print()

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


Fine-tuned model responses:

Input:    Customer says: I ordered the wrong size


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: 
A below below Below Below Below Below Below below below below Below Below Below Below Below Below are below Below any practical Below Below Below Below Below Below Below Below Below Below Below are b

Input:    Customer says: Where is my invoice?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: 
As Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below below Below Below Below Below Below Belo

Input:    Customer says: The color looks different from the website photo
Response: 
A program Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below Below could below Below Below Below Below Below Below below Below Bel



In [11]:
# Make the dataset use Llama's chat template
def format_example(example):
    messages = [
        {"role": "system", "content": "You are a helpful customer-support assistant."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = Dataset.from_list(training_examples).map(format_example)

Map: 100%|██████████| 8/8 [00:00<00:00, 575.41 examples/s]


In [12]:
from trl import SFTConfig, SFTTrainer

model.config.use_cache = False  # required during checkpointed training

training_args = SFTConfig(
    output_dir="./fine_tune_output",
    num_train_epochs=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

Building labels for train dataset: 100%|██████████| 8/8 [00:00<00:00, 3512.45 examples/s]


Step,Training Loss
1,2.493347
2,1.896711
3,2.073864
4,1.710909
5,1.388692
6,1.097995
7,1.486596
8,1.459517
9,1.229609
10,1.187907


TrainOutput(global_step=160, training_loss=0.19901995982509108, metrics={'train_runtime': 120.7051, 'train_samples_per_second': 1.326, 'train_steps_per_second': 1.326, 'total_flos': 99644706570240.0, 'train_loss': 0.19901995982509108, 'epoch': 20.0})

In [14]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

In [15]:
# Convert examples to the model's native chat format
def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful, concise customer-support assistant."
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        },
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = Dataset.from_list(training_examples).map(format_example)

print(dataset[0]["text"])

Map: 100%|██████████| 8/8 [00:00<00:00, 1295.29 examples/s]

<|system|>
You are a helpful, concise customer-support assistant.</s>
<|user|>
Customer says: My order hasn't arrived yet. Order #12345</s>
<|assistant|>
I apologize for the delay. Let me check order #12345 for you right away. Could you please confirm your email address so I can pull up the details and provide you with an accurate update on your delivery status?</s>



In [16]:
from trl import SFTConfig, SFTTrainer

# Required when gradient checkpointing is enabled during training
model.config.use_cache = False

training_args = SFTConfig(
    output_dir="./fine_tune_output",
    num_train_epochs=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("✅ Trainer ready")

Building labels for train dataset: 100%|██████████| 8/8 [00:00<00:00, 179.98 examples/s]

✅ Trainer ready


In [17]:
print("Starting QLoRA fine-tuning...")
train_result = trainer.train()

print(f"✅ Training complete")
print(train_result.metrics)

Starting QLoRA fine-tuning...


Step,Training Loss
1,0.334591
2,0.201288
3,0.206357
4,0.105940
5,0.042136
6,0.036355
7,0.033253
8,0.032598
9,0.021251
10,0.027603


✅ Training complete
{'train_runtime': 118.0017, 'train_samples_per_second': 1.356, 'train_steps_per_second': 1.356, 'total_flos': 102660435394560.0, 'train_loss': 0.029202523914864286, 'epoch': 20.0}


In [18]:
adapter_path = "./customer_support_lora"

trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"✅ LoRA adapter saved to: {adapter_path}")

✅ LoRA adapter saved to: ./customer_support_lora


In [19]:
# Switch from training mode to inference mode
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

print("✅ Model ready for inference")

✅ Model ready for inference


In [21]:
def get_response(customer_message):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful, concise customer-support assistant."
        },
        {
            "role": "user",
            "content": f"Customer says: {customer_message}"
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            inputs,
            max_new_tokens=120,  # Don't pass max_length here
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    new_tokens = output[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


test_inputs = [
    "I ordered the wrong size",
    "Where is my invoice?",
    "The color looks different from the website photo",
]

for item in test_inputs:
    print(f"\nInput:    Customer says: {item}")
    print(f"Response: {get_response(item)}")


Input:    Customer says: I ordered the wrong size


AttributeError: 